# 15-minute LSTM training — agent-final

Trains **Single / Double / BiLSTM** on clean preprocessed 15min data.

**Before this:** run `preprocess.ipynb` → saves `outputs/preprocess/15min/data.csv`

**No SHAP here** — XAI comes later.

In [ ]:
USE_COLAB = False
FORCE_RETRAIN = False

window_sizes = [1, 4, 8, 16, 24, 48, 64, 96, 672]
epochs = 50
batch_size = 32
test_ratio = 0.18

if USE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Shared-Colab-Storage/agent-final"
else:
    from pathlib import Path
    BASE = Path("..").resolve()

DATA_CSV = f"{BASE}/outputs/preprocess/15min/data.csv"
SCALER_PKL = f"{BASE}/outputs/preprocess/15min/scaler.pkl"
SAVE_PATH = f"{BASE}/outputs/train/15min"

print("DATA:", DATA_CSV)
print("SAVE:", SAVE_PATH)

In [ ]:
if USE_COLAB:
    !pip install -q pandas numpy scikit-learn matplotlib tensorflow

import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.models import Sequential, load_model

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_true - y_pred)))

os.makedirs(SAVE_PATH, exist_ok=True)

## 1) Load preprocessed 15min data

In [ ]:
scaled_df = pd.read_csv(DATA_CSV)
with open(SCALER_PKL, "rb") as f:
    scaler = pickle.load(f)

feature_cols = list(scaled_df.columns)
target_idx = 0
n_features = len(feature_cols)

print("Rows:", len(scaled_df))
print("Features:", feature_cols)
scaled_df.head()

## 2) Sliding windows

In [ ]:
arr = scaled_df[feature_cols].to_numpy(dtype=np.float32)

data = {}
for window in window_sizes:
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window])
        y.append(arr[i + window, target_idx])
    X, y = np.array(X), np.array(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_ratio, shuffle=False
    )
    data[f"win{window}"] = dict(
        X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test
    )
    print(f"win{window}: train {X_train.shape}, test {X_test.shape}")

## 3) Train models

In [ ]:
stacks = ["single", "double", "bidir"]
trained = {}

for stack in stacks:
    print(f"\n========== {stack.upper()} LSTM ==========")
    model_dir = os.path.join(SAVE_PATH, stack, "models")
    hist_dir = os.path.join(SAVE_PATH, stack, "history")
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)
    trained[stack] = {}

    for window in window_sizes:
        model_path = os.path.join(model_dir, f"win{window}.keras")
        hist_path = os.path.join(hist_dir, f"win{window}.pkl")
        d = data[f"win{window}"]

        if os.path.exists(model_path) and not FORCE_RETRAIN:
            model = load_model(model_path, custom_objects={"rmse": rmse})
            hist = pickle.load(open(hist_path, "rb")) if os.path.exists(hist_path) else {}
            print(f"  win{window}: loaded")
            trained[stack][window] = {"model": model, "history": hist}
            continue

        print(f"  win{window}: training...")
        model = Sequential()
        if stack == "single":
            model.add(LSTM(64, input_shape=(window, n_features)))
        elif stack == "double":
            model.add(LSTM(64, return_sequences=True, input_shape=(window, n_features)))
            model.add(LSTM(64))
        else:
            model.add(Bidirectional(LSTM(64), input_shape=(window, n_features)))
        model.add(Dropout(0.1))
        model.add(Dense(1))
        model.compile(optimizer="adam", loss=rmse, metrics=["mae"])

        hist = model.fit(
            d["X_train"], d["y_train"],
            validation_data=(d["X_test"], d["y_test"]),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1,
        )
        model.save(model_path)
        pickle.dump(hist.history, open(hist_path, "wb"))
        trained[stack][window] = {"model": model, "history": hist.history}
        print(f"  win{window}: saved")

## 4) Evaluate

In [ ]:
def to_kwh(values):
    values = np.asarray(values).ravel()
    d = np.zeros((len(values), n_features))
    d[:, target_idx] = values
    return scaler.inverse_transform(d)[:, target_idx]

rows = []
for stack in stacks:
    for window in window_sizes:
        if window not in trained[stack]:
            continue
        model = trained[stack][window]["model"]
        y_test = data[f"win{window}"]["y_test"]
        y_pred = model.predict(data[f"win{window}"]["X_test"], verbose=0).ravel()

        yt, yp = to_kwh(y_test), to_kwh(y_pred)
        rmse_k = float(np.sqrt(mean_squared_error(yt, yp)))
        mae_k = float(mean_absolute_error(yt, yp))
        r2_k = float(r2_score(yt, yp))
        mean_obs = np.mean(yt)
        wia_k = float(1 - np.sum((yp - yt) ** 2) / np.sum((np.abs(yp - mean_obs) + np.abs(yt - mean_obs)) ** 2))

        rows.append({
            "model": stack, "window": window,
            "rmse_kwh": rmse_k, "mae_kwh": mae_k, "r2": r2_k, "wia": wia_k,
        })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(SAVE_PATH, "results_metrics.csv"), index=False)

best = metrics_df.loc[metrics_df["rmse_kwh"].idxmin()]
print("Best model:")
print(best)
metrics_df.sort_values("rmse_kwh").head(10)

## 5) RMSE heatmap

In [ ]:
pivot = metrics_df.pivot(index="window", columns="model", values="rmse_kwh")
plt.figure(figsize=(6, 7))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd_r")
plt.title("15min — RMSE (kWh) by window & architecture")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "rmse_heatmap.png"), dpi=120)
plt.show()

## Done

Saved under `outputs/train/15min/`:
- models, `results_metrics.csv`, `rmse_heatmap.png`